In [8]:
import numpy as np
import scipy.io as sio
import pandas as pd
import os
from pathlib import Path
import subprocess
from tqdm import tqdm
from datetime import datetime

# # CIMT_rest setup
# base_dir = Path("HMM/CIMT_rest/results/ICA_14c_no_TDE/7_states/inf_params/omst_filter_matrices")
# k_value = 7
# stim_dir = base_dir / "stim_matrices"
# resting_dir = base_dir / "resting_matrices"
# stim_metrics_dir = base_dir / "stim_metrics"
# resting_metrics_dir = base_dir / "resting_metrics"
# reports_dir = base_dir / "reports"

# # Define paths for stim matrices (k5 through k15)
# matrix_stim_paths = [
    
# ]

# # Define paths for resting state matrices (npz files)
# matrix_resting_paths = [
#     "HMM/CIMT_rest/results/ICA_14c_no_TDE/7_states/inf_params/all_corrs_7_states.npz"
# ]

# CIMT_task setup
base_dir = Path("HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/omst_filter_matrices")
k_value = 4
stim_dir = base_dir / "stim_matrices"
resting_dir = base_dir / "resting_matrices"
stim_metrics_dir = base_dir / "stim_metrics"
resting_metrics_dir = base_dir / "resting_metrics"
reports_dir = base_dir / "reports"

# Define paths for stim matrices (k5 through k15)
matrix_stim_paths = [
    "HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/all_corrs_4_states.npz"
]

# Define paths for resting state matrices (npz files)
matrix_resting_paths = [
    
]

# # FPI setup
# base_dir = Path("HMM/FPI/results/ICA_13c_no_TDE/11_states/inf_params/omst_filter_matrices")
# k_value = 
# stim_dir = base_dir / "stim_matrices"
# resting_dir = base_dir / "resting_matrices"
# stim_metrics_dir = base_dir / "stim_metrics"
# resting_metrics_dir = base_dir / "resting_metrics"
# reports_dir = base_dir / "reports"

# # Define paths for stim matrices (k5 through k15)
# matrix_stim_paths = [
    
# ]

# # Define paths for resting state matrices (npz files)
# matrix_resting_paths = [
#     "HMM/FPI/results/ICA_13c_no_TDE/11_states/inf_params/all_corrs_11_states.npz"
# ]

# Create directories if they don't exist
for dir_path in [base_dir, stim_dir, resting_dir, stim_metrics_dir, resting_metrics_dir, reports_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

def create_matlab_script(input_file, output_file, script_name="temp_omst_script.m"):
    """
    Create a temporary MATLAB script that calls the OMST function
    """
    script_content = f"""
% Temporary MATLAB script for OMST processing
fprintf('Starting OMST processing...\\n');
try
    % Add path to OMST function
    addpath('/Users/fei/Documents/MATLAB/toolbox/topological_filtering_networks-master/threshold_schemes/threshold_schemes');
    
    % Call the OMST function
    run_omst_single_matrix('{input_file}', '{output_file}');
    
    fprintf('OMST processing completed successfully\\n');
    exit(0);
catch ME
    fprintf('Error in OMST processing: %s\\n', ME.message);
    fprintf('Stack trace:\\n');
    for i = 1:length(ME.stack)
        fprintf('  File: %s, Line: %d\\n', ME.stack(i).file, ME.stack(i).line);
    end
    exit(1);
end
"""
    
    with open(script_name, 'w') as f:
        f.write(script_content)
    
    return script_name

def process_omst_matlab(centroid_array, k_value, data_type, matlab_exe="matlab"):
    """
    Process a single centroid array through MATLAB OMST filtering using subprocess
    """
    
    # IMPORTANT: Convert to double precision (float64)
    centroid_array = centroid_array.astype(np.float64)
    
    # Create temporary files for MATLAB I/O
    temp_input = f"temp_input_k{k_value}_{data_type}.mat"
    temp_output = f"temp_output_k{k_value}_{data_type}.mat"
    temp_script = f"temp_script_k{k_value}_{data_type}.m"
    
    # Save centroid array to .mat file
    print(f"  Saving input data to {temp_input}...")
    print(f"  Data type: {centroid_array.dtype}, Shape: {centroid_array.shape}")
    
    # Save with explicit v7.3 format for better compatibility
    sio.savemat(temp_input, {'centroid_coherence_array': centroid_array}, do_compression=False)
    
    # Create MATLAB script
    script_name = create_matlab_script(temp_input, temp_output, temp_script)
    
    # Run MATLAB
    print(f"  Running MATLAB OMST processing...")
    try:
        # For Windows
        if os.name == 'nt':
            cmd = [matlab_exe, "-batch", f"run('{script_name}')"]
        else:  # For Linux/Mac
            cmd = ["/usr/bin/arch", "-arm64", matlab_exe, "-batch", f"run('{script_name}')"]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=600  # 10 minute timeout
        )
        
        if result.returncode != 0:
            print(f"  MATLAB error (return code {result.returncode}):")
            print(f"  STDOUT: {result.stdout}")
            print(f"  STDERR: {result.stderr}")
            return None
        else:
            print(f"  MATLAB processing successful")
            
    except subprocess.TimeoutExpired:
        print(f"  MATLAB timeout for k={k_value} {data_type}")
        return None
    except FileNotFoundError:
        print(f"  MATLAB executable not found. Please check the path: {matlab_exe}")
        print("  You may need to specify the full path to MATLAB, e.g.:")
        print("  Windows: 'C:/Program Files/MATLAB/R2023b/bin/matlab.exe'")
        print("  Mac: '/Applications/MATLAB_R2023b.app/bin/matlab'")
        print("  Linux: '/usr/local/MATLAB/R2023b/bin/matlab'")
        return None
    except Exception as e:
        print(f"  Error running MATLAB: {e}")
        return None
    finally:
        # Clean up script file
        if os.path.exists(script_name):
            os.remove(script_name)
    
    # Load results from MATLAB
    if os.path.exists(temp_output):
        print(f"  Loading results from {temp_output}...")
        results = sio.loadmat(temp_output)
        
        # Clean up temporary files
        os.remove(temp_input)
        os.remove(temp_output)
        
        return results
    else:
        print(f"  Output file not created for k={k_value} {data_type}")
        if os.path.exists(temp_input):
            os.remove(temp_input)
        return None

def extract_metrics_from_ov(ov_data, k_value):
    """
    Extract metrics from MATLAB ov cell array structure
    """
    metrics_list = []
    
    # Handle MATLAB cell array structure
    if isinstance(ov_data, np.ndarray):
        num_clusters = len(ov_data)
        
        for cluster_idx in range(num_clusters):
            # Access the struct for this cluster
            cluster_data = ov_data[cluster_idx]
            
            # Initialize metrics dict
            metrics = {
                'k_value': k_value,
                'cluster': cluster_idx + 1,
                'nCIJtree': np.nan,
                'mdeg': np.nan,
                'gce': np.nan,
                'costmax': np.nan,
                'E': np.nan
            }
            
            # Extract metrics from the struct - handle different possible structures
            try:
                if isinstance(cluster_data, np.ndarray) and cluster_data.size > 0:
                    # Try to access the structured array
                    if cluster_data.ndim > 1:
                        struct_data = cluster_data[0, 0]
                    else:
                        struct_data = cluster_data.item()
                    
                    # Try to extract each field with various access patterns
                    if hasattr(struct_data, 'dtype') and struct_data.dtype.names:
                        # It's a structured array
                        for field in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                            if field in struct_data.dtype.names:
                                try:
                                    value = struct_data[field]
                                    # Handle different shapes of the value
                                    if isinstance(value, np.ndarray):
                                        if value.size == 1:
                                            metrics[field] = float(value.flatten()[0])
                                        else:
                                            metrics[field] = float(value[0, 0]) if value.ndim > 1 else float(value[0])
                                    else:
                                        metrics[field] = float(value)
                                except (IndexError, TypeError, ValueError):
                                    pass  # Keep as np.nan
                    elif isinstance(struct_data, dict):
                        # It's a dictionary
                        for field in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                            if field in struct_data:
                                try:
                                    value = struct_data[field]
                                    if isinstance(value, np.ndarray):
                                        if value.size == 1:
                                            metrics[field] = float(value.flatten()[0])
                                        else:
                                            metrics[field] = float(value[0, 0]) if value.ndim > 1 else float(value[0])
                                    else:
                                        metrics[field] = float(value)
                                except (IndexError, TypeError, ValueError):
                                    pass  # Keep as np.nan
            except Exception as e:
                print(f"    Warning: Could not extract metrics for cluster {cluster_idx + 1}: {e}")
            
            metrics_list.append(metrics)
    
    return pd.DataFrame(metrics_list)

def save_omst_results_with_metrics(results, k_value, save_dir, metrics_dir, data_type):
    """
    Save OMST results as numpy files and metrics as CSV files
    """
    
    # Save Strategy A - Positive edges
    np.save(save_dir / f"k{k_value}_strategy_A_positive.npy", 
            results['omst_pos_centroid_A'])
    
    # Save Strategy A - Negative edges
    np.save(save_dir / f"k{k_value}_strategy_A_negative.npy", 
            results['omst_neg_centroid_A'])
    
    # Save Strategy B - Combined
    np.save(save_dir / f"k{k_value}_strategy_B.npy", 
            results['omst_centroid_B'])
    
    print(f"  Saved OMST matrices for k={k_value} to {save_dir}")
    
    all_metrics = []
    
    # Extract and save metrics for Strategy A - Positive
    if 'omst_centroid_A_ov_pos' in results:
        try:
            df_pos = extract_metrics_from_ov(results['omst_centroid_A_ov_pos'], k_value)
            df_pos['strategy'] = 'A_positive'
            csv_path = metrics_dir / f"k{k_value}_strategy_A_positive_metrics.csv"
            df_pos.to_csv(csv_path, index=False)
            print(f"  Saved Strategy A positive metrics to {csv_path}")
            all_metrics.append(df_pos)
        except Exception as e:
            print(f"  Warning: Could not extract Strategy A positive metrics: {e}")
    
    # Extract and save metrics for Strategy A - Negative
    if 'omst_centroid_A_ov_neg' in results:
        try:
            df_neg = extract_metrics_from_ov(results['omst_centroid_A_ov_neg'], k_value)
            df_neg['strategy'] = 'A_negative'
            csv_path = metrics_dir / f"k{k_value}_strategy_A_negative_metrics.csv"
            df_neg.to_csv(csv_path, index=False)
            print(f"  Saved Strategy A negative metrics to {csv_path}")
            all_metrics.append(df_neg)
        except Exception as e:
            print(f"  Warning: Could not extract Strategy A negative metrics: {e}")
    
    # Extract and save metrics for Strategy B
    if 'omst_centroid_B_ov' in results:
        try:
            df_b = extract_metrics_from_ov(results['omst_centroid_B_ov'], k_value)
            df_b['strategy'] = 'B'
            csv_path = metrics_dir / f"k{k_value}_strategy_B_metrics.csv"
            df_b.to_csv(csv_path, index=False)
            print(f"  Saved Strategy B metrics to {csv_path}")
            all_metrics.append(df_b)
        except Exception as e:
            print(f"  Warning: Could not extract Strategy B metrics: {e}")
    
    # Combine all metrics for summary
    if all_metrics:
        combined_df = pd.concat(all_metrics, ignore_index=True)
        combined_csv_path = metrics_dir / f"k{k_value}_all_metrics.csv"
        combined_df.to_csv(combined_csv_path, index=False)
        print(f"  Saved combined metrics to {combined_csv_path}")
        return combined_df
    
    return None

def generate_text_report(all_results, report_path):
    """
    Generate a comprehensive text report of all OMST processing results
    """
    
    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("OMST FILTERING RESULTS REPORT\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        
        # Separate by data type
        stim_results = {k: v for k, v in all_results.items() if 'stim' in k}
        resting_results = {k: v for k, v in all_results.items() if 'resting' in k}
        
        # Report for STIM data
        if stim_results:
            f.write("STIM DATA RESULTS\n")
            f.write("-"*40 + "\n\n")
            
            for key in sorted(stim_results.keys()):
                df = stim_results[key]
                if df is not None and not df.empty:
                    k_value = df['k_value'].iloc[0]
                    f.write(f"K={k_value} STIM Results:\n")
                    f.write("-"*20 + "\n")
                    
                    # Group by strategy
                    for strategy in df['strategy'].unique():
                        strategy_df = df[df['strategy'] == strategy]
                        f.write(f"\nStrategy: {strategy}\n")
                        f.write(f"Number of clusters: {len(strategy_df)}\n")
                        
                        # Calculate summary statistics
                        f.write("\nSummary Statistics:\n")
                        for metric in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                            if metric in strategy_df.columns:
                                values = strategy_df[metric].dropna()
                                if len(values) > 0:
                                    f.write(f"  {metric:10s}: mean={values.mean():8.4f}, "
                                          f"std={values.std():8.4f}, "
                                          f"min={values.min():8.4f}, "
                                          f"max={values.max():8.4f}\n")
                        
                        # Detailed cluster information
                        f.write("\nDetailed Cluster Metrics:\n")
                        for _, row in strategy_df.iterrows():
                            f.write(f"  Cluster {int(row['cluster'])}: ")
                            for metric in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                                if not pd.isna(row[metric]):
                                    f.write(f"{metric}={row[metric]:.4f}, ")
                            f.write("\n")
                    
                    f.write("\n" + "="*40 + "\n\n")
        
        # Report for RESTING data
        if resting_results:
            f.write("\nRESTING DATA RESULTS\n")
            f.write("-"*40 + "\n\n")
            
            for key in sorted(resting_results.keys()):
                df = resting_results[key]
                if df is not None and not df.empty:
                    k_value = df['k_value'].iloc[0]
                    f.write(f"K={k_value} RESTING Results:\n")
                    f.write("-"*20 + "\n")
                    
                    # Group by strategy
                    for strategy in df['strategy'].unique():
                        strategy_df = df[df['strategy'] == strategy]
                        f.write(f"\nStrategy: {strategy}\n")
                        f.write(f"Number of clusters: {len(strategy_df)}\n")
                        
                        # Calculate summary statistics
                        f.write("\nSummary Statistics:\n")
                        for metric in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                            if metric in strategy_df.columns:
                                values = strategy_df[metric].dropna()
                                if len(values) > 0:
                                    f.write(f"  {metric:10s}: mean={values.mean():8.4f}, "
                                          f"std={values.std():8.4f}, "
                                          f"min={values.min():8.4f}, "
                                          f"max={values.max():8.4f}\n")
                        
                        # Detailed cluster information
                        f.write("\nDetailed Cluster Metrics:\n")
                        for _, row in strategy_df.iterrows():
                            f.write(f"  Cluster {int(row['cluster'])}: ")
                            for metric in ['nCIJtree', 'mdeg', 'gce', 'costmax', 'E']:
                                if not pd.isna(row[metric]):
                                    f.write(f"{metric}={row[metric]:.4f}, ")
                            f.write("\n")
                    
                    f.write("\n" + "="*40 + "\n\n")
        
        f.write("\nEND OF REPORT\n")
        f.write("="*80 + "\n")

# IMPORTANT: Set your MATLAB path here
matlab_path = "/Applications/MATLAB_R2024b.app/bin/matlab"  # Change this to your MATLAB installation path if needed

print("="*60)
print("OMST FILTERING PIPELINE")
print("="*60)

# MATLAB function with explicit double conversion (same as before)
matlab_function_code = """function run_omst_single_matrix(input_file, output_file)
    % run_omst_single_matrix.m
    %
    % Process a single 3D centroid coherence matrix with OMST filtering
    
    fprintf('Loading %s...\\n', input_file);
    data = load(input_file);
    
    % ENSURE DOUBLE PRECISION
    cent_coh_mat = double(data.centroid_coherence_array);
    
    original_size = size(cent_coh_mat);
    fprintf('Original centroid coherence array size: [%d %d %d]\\n', original_size(1), original_size(2), original_size(3));
    fprintf('Data type: %s\\n', class(cent_coh_mat));
    
    if original_size(1) < original_size(2) && original_size(2) == original_size(3)
        cent_coh_mat = permute(cent_coh_mat, [2, 3, 1]);
        fprintf('Permuted from (K,N,N) to (N,N,K)\\n');
    end
    
    s_cent = size(cent_coh_mat);
    N = s_cent(1);
    K = s_cent(3);
    
    fprintf('Matrix dimensions: N=%d ROIs, K=%d clusters\\n', N, K);
    
    % Preallocate
    omst_pos_centroid_A = zeros(N, N, K);
    omst_neg_centroid_A = zeros(N, N, K);
    omst_centroid_A_ov_pos = cell(K,1);
    omst_centroid_A_ov_neg = cell(K,1);
    omst_centroid_B = zeros(N, N, K);
    omst_centroid_B_ov = cell(K,1);
    
    epsilon = 1e-12;
    
    % Strategy A
    fprintf('\\nProcessing Strategy A...\\n');
    for i = 1:K
        fprintf('  Cluster %d/%d\\n', i, K);
        mtx_original = double(cent_coh_mat(:,:,i));  % Ensure double
        
        % Positive
        pos_mtx = mtx_original;
        pos_mtx(pos_mtx < 0) = 0;
        pos_mtx(pos_mtx == 0) = epsilon;
        pos_mtx = double(pos_mtx);  % Ensure double before OMST
        
        [nCIJtree_pos, CIJtree_pos, mdeg_pos, gce_pos, costmax_pos, E_pos] = ...
            threshold_omst_gce_wu_very_fast(pos_mtx, 0);
        
        omst_pos_centroid_A(:,:,i) = CIJtree_pos;
        omst_centroid_A_ov_pos{i} = struct('nCIJtree', nCIJtree_pos, ...
                                          'mdeg', mdeg_pos, ...
                                          'gce', gce_pos, ...
                                          'costmax', costmax_pos, ...
                                          'E', E_pos);
        
        % Negative
        neg_mtx = mtx_original;
        neg_mtx(neg_mtx > 0) = 0;
        neg_mtx = abs(neg_mtx);
        neg_mtx(neg_mtx == 0) = epsilon;
        neg_mtx = double(neg_mtx);  % Ensure double before OMST
        
        [nCIJtree_neg, CIJtree_neg, mdeg_neg, gce_neg, costmax_neg, E_neg] = ...
            threshold_omst_gce_wu_very_fast(neg_mtx, 0);
        
        omst_neg_centroid_A(:,:,i) = -CIJtree_neg;
        omst_centroid_A_ov_neg{i} = struct('nCIJtree', nCIJtree_neg, ...
                                          'mdeg', mdeg_neg, ...
                                          'gce', gce_neg, ...
                                          'costmax', costmax_neg, ...
                                          'E', E_neg);
    end
    
    % Strategy B
    fprintf('\\nProcessing Strategy B...\\n');
    for i = 1:K
        fprintf('  Cluster %d/%d\\n', i, K);
        mtx_original = double(cent_coh_mat(:,:,i));  % Ensure double
        
        abs_mtx = abs(mtx_original);
        abs_mtx(abs_mtx == 0) = epsilon;
        abs_mtx = double(abs_mtx);  % Ensure double before OMST
        
        [nCIJtree_abs, CIJtree_abs, mdeg_abs, gce_abs, costmax_abs, E_abs] = ...
            threshold_omst_gce_wu_very_fast(abs_mtx, 0);
        
        sign_mtx = sign(mtx_original);
        CIJtree_signed = CIJtree_abs .* sign_mtx;
        
        omst_centroid_B(:,:,i) = CIJtree_signed;
        omst_centroid_B_ov{i} = struct('nCIJtree', nCIJtree_abs, ...
                                      'mdeg', mdeg_abs, ...
                                      'gce', gce_abs, ...
                                      'costmax', costmax_abs, ...
                                      'E', E_abs);
    end
    
    % Transpose back to (K, N, N) for Python
    omst_pos_centroid_A = permute(omst_pos_centroid_A, [3, 1, 2]);
    omst_neg_centroid_A = permute(omst_neg_centroid_A, [3, 1, 2]);
    omst_centroid_B = permute(omst_centroid_B, [3, 1, 2]);
    
    fprintf('\\nSaving results to %s...\\n', output_file);
    save(output_file, ...
        'omst_pos_centroid_A', 'omst_neg_centroid_A', ...
        'omst_centroid_A_ov_pos', 'omst_centroid_A_ov_neg', ...
        'omst_centroid_B', 'omst_centroid_B_ov', ...
        'N', 'K', ...
        '-v7');
    
    fprintf('Results saved successfully.\\n');
    fprintf('Final matrix dimensions: %d x %d x %d (K x N x N)\\n', K, N, N);
end"""

# Save the MATLAB function
with open("run_omst_single_matrix.m", "w") as f:
    f.write(matlab_function_code)
print("Saved MATLAB function: run_omst_single_matrix.m")

# Dictionary to store all metrics for report generation
all_metrics_results = {}

# Process STIM matrices
print("\n" + "="*50)
print("Processing STIM matrices...")
print("="*50)

for i, stim_path in enumerate(matrix_stim_paths):
    print(f"\nProcessing k={k_value} STIM matrix...")
    
    # Load the file
    try:
        if stim_path.endswith('.npz'):
            data = np.load(stim_path)
            print(f"  Available keys in npz: {list(data.keys())}")
            
            if 'centroid_coherence_array' in data:
                centroid_array = data['centroid_coherence_array']
            elif 'centroid_coherence_arrays' in data:
                centroid_array = data['centroid_coherence_arrays']
            else:
                for key in data.keys():
                    if isinstance(data[key], np.ndarray) and data[key].ndim == 3:
                        centroid_array = data[key]
                        print(f"  Using key: {key}")
                        break
                else:
                    print(f"  No 3D array found in {stim_path}")
                    continue
        else:
            centroid_array = np.load(stim_path)
            
        print(f"  Loaded array shape: {centroid_array.shape}, dtype: {centroid_array.dtype}")
        
        if centroid_array.shape[0] != k_value:
            print(f"  WARNING: Array has {centroid_array.shape[0]} clusters but expected k={k_value}")
        
        if centroid_array.dtype != np.float64:
            print(f"  Converting from {centroid_array.dtype} to float64...")
            centroid_array = centroid_array.astype(np.float64)
            
    except Exception as e:
        print(f"  Error loading {stim_path}: {e}")
        continue
    
    # Process through MATLAB
    results = process_omst_matlab(centroid_array, k_value, 'stim', matlab_path)
    
    if results is not None:
        # Save results with metrics
        metrics_df = save_omst_results_with_metrics(results, k_value, stim_dir, stim_metrics_dir, 'stim')
        if metrics_df is not None:
            all_metrics_results[f'stim_k{k_value}'] = metrics_df

# Process RESTING STATE matrices
print("\n" + "="*50)
print("Processing RESTING STATE matrices...")
print("="*50)

for i, resting_path in enumerate(matrix_resting_paths):
    print(f"\nProcessing k={k_value} RESTING matrix...")
    
    # Load the npz file and extract centroid array
    try:
        data = np.load(resting_path)
        print(f"  Available keys in npz: {list(data.keys())}")
        
        if 'centroid_matrices' in data:
            centroid_array = data['centroid_matrices']
        elif 'centroids' in data:
            centroid_array = data['centroids']
        elif 'centroid_coherence_array' in data:
            centroid_array = data['centroid_coherence_array']
        else:
            possible_keys = [k for k in data.keys() if 'centroid' in k.lower()]
            if possible_keys:
                centroid_array = data[possible_keys[0]]
                print(f"  Using key: {possible_keys[0]}")
            else:
                print(f"  Could not find centroid matrix in {resting_path}")
                continue
        
        print(f"  Loaded array shape: {centroid_array.shape}, dtype: {centroid_array.dtype}")
        
        if centroid_array.dtype != np.float64:
            print(f"  Converting from {centroid_array.dtype} to float64...")
            centroid_array = centroid_array.astype(np.float64)
            
    except Exception as e:
        print(f"  Error loading {resting_path}: {e}")
        continue
    
    # Process through MATLAB
    results = process_omst_matlab(centroid_array, k_value, 'resting', matlab_path)
    
    if results is not None:
        # Save results with metrics
        metrics_df = save_omst_results_with_metrics(results, k_value, resting_dir, resting_metrics_dir, 'resting')
        if metrics_df is not None:
            all_metrics_results[f'resting_k{k_value}'] = metrics_df

# Generate comprehensive text report
print("\n" + "="*50)
print("Generating comprehensive report...")
print("="*50)

report_path = reports_dir / f"omst_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
generate_text_report(all_metrics_results, report_path)
print(f"Report saved to: {report_path}")

# Also save a combined CSV of all metrics
if all_metrics_results:
    all_dfs = []
    for key, df in all_metrics_results.items():
        df['data_type'] = 'stim' if 'stim' in key else 'resting'
        all_dfs.append(df)
    
    combined_all = pd.concat(all_dfs, ignore_index=True)
    combined_csv_path = base_dir / f"all_metrics_combined_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    combined_all.to_csv(combined_csv_path, index=False)
    print(f"Combined metrics CSV saved to: {combined_csv_path}")

print("\n" + "="*50)
print("Processing complete!")
print("="*50)
print(f"\nResults saved in:")
print(f"  - Matrices: {stim_dir} and {resting_dir}")
print(f"  - Metrics: {stim_metrics_dir} and {resting_metrics_dir}")
print(f"  - Reports: {reports_dir}")

# Verify saved files
print("\nVerifying saved files...")
print("\nSTIM matrices:")
for file in sorted(stim_dir.glob("*.npy")):
    arr = np.load(file)
    print(f"  {file.name}: shape {arr.shape}")

print("\nRESTING matrices:")
for file in sorted(resting_dir.glob("*.npy")):
    arr = np.load(file)
    print(f"  {file.name}: shape {arr.shape}")

print("\nSTIM metrics:")
for file in sorted(stim_metrics_dir.glob("*.csv")):
    print(f"  {file.name}")

print("\nRESTING metrics:")
for file in sorted(resting_metrics_dir.glob("*.csv")):
    print(f"  {file.name}")

OMST FILTERING PIPELINE
Saved MATLAB function: run_omst_single_matrix.m

Processing STIM matrices...

Processing k=4 STIM matrix...
  Available keys in npz: ['centroids']
  Using key: centroids
  Loaded array shape: (4, 114, 114), dtype: float64
  Saving input data to temp_input_k4_stim.mat...
  Data type: float64, Shape: (4, 114, 114)
  Running MATLAB OMST processing...
  MATLAB processing successful
  Loading results from temp_output_k4_stim.mat...
  Saved OMST matrices for k=4 to HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/omst_filter_matrices/stim_matrices
  Saved Strategy A positive metrics to HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/omst_filter_matrices/stim_metrics/k4_strategy_A_positive_metrics.csv
  Saved Strategy A negative metrics to HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/omst_filter_matrices/stim_metrics/k4_strategy_A_negative_metrics.csv
  Saved Strategy B metrics to HMM/CIMT_task/results/ICA_14c_no_TDE/4_states/inf_params/oms